# Notebook 01 — Read & Join the Tables

## Objective

The goal of this notebook is to read the Olist tables from PostgreSQL, inspect their structure and relationships, aggregate tables with multiple rows per order, and create a single machine-learning table with **one row per order**.

### Output Artifact

`artifacts/01_joined/ml_orders.parquet`

## 1. Setup

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, URL

In [2]:
import pandas as pd
import pyarrow as pa

print("pandas:", pd.__version__)
print("pyarrow:", pa.__version__)

pandas: 3.0.5
pyarrow: 25.0.1


In [3]:
test_df = pd.DataFrame({
    "a": [1, 2, 3],
    "b": ["x", "y", "z"]
})

test_df.to_parquet("test.parquet", index=False)

print("Parquet test successful")

Parquet test successful


In [4]:
PROJECT_ROOT = Path.cwd().parent

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "01_joined"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

ARTIFACT_PATH = ARTIFACT_DIR / "ml_orders.parquet"

ARTIFACT_PATH

WindowsPath('C:/Users/CW/Documents/Projects/Dataset_Olist/artifacts/01_joined/ml_orders.parquet')

In [5]:
# Connect to PostgreSQL 
load_dotenv(PROJECT_ROOT / ".env")

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("POSTGRES_HOST"),
    port=int(os.getenv("POSTGRES_PORT")),
    database=os.getenv("POSTGRES_DB"),
)

engine = create_engine(db_url)

In [6]:
pd.read_sql("SELECT 1 AS connection_test;", engine)

,connection_test
0,1


## 2. Read the Database Tables

Load all Olist tables from PostgreSQL into pandas DataFrames.

In [7]:
table_names = [
    "customers",
    "orders",
    "order_items",
    "products",
    "sellers",
    "order_payments",
    "order_reviews",
    "geolocation",
    "product_category_translation",
]

tables = {}

for table_name in table_names:
    tables[table_name] = pd.read_sql(
        f"SELECT * FROM {table_name}",
        engine
    )

    print(
        f"{table_name:<30} "
        f"rows={tables[table_name].shape[0]:>8,} "
        f"columns={tables[table_name].shape[1]:>2}"
    )

customers                      rows=  99,441 columns= 5
orders                         rows=  99,441 columns= 8
order_items                    rows= 112,650 columns= 7
products                       rows=  32,951 columns= 9
sellers                        rows=   3,095 columns= 4
order_payments                 rows= 103,886 columns= 5
order_reviews                  rows=  99,224 columns= 7
geolocation                    rows=1,000,163 columns= 5
product_category_translation   rows=      71 columns= 2


In [8]:
customers = tables["customers"]
orders = tables["orders"]
order_items = tables["order_items"]
products = tables["products"]
sellers = tables["sellers"]
order_payments = tables["order_payments"]
order_reviews = tables["order_reviews"]
geolocation = tables["geolocation"]
category_translation = tables["product_category_translation"]

## 3. Inspect Each Table

Before joining the tables, inspect their shapes, columns, and sample rows to understand what one row represents in each dataset.


In [9]:
for name, df in tables.items():
    print(f"\n{name.upper()}")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    display(df.head(3))


CUSTOMERS
Shape: (99441, 5)
Columns: ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



ORDERS
Shape: (99441, 8)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04



ORDER_ITEMS
Shape: (112650, 7)
Columns: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


...

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24



GEOLOCATION
Shape: (1000163, 5)
Columns: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP



PRODUCT_CATEGORY_TRANSLATION
Shape: (71, 2)
Columns: ['product_category_name', 'product_category_name_english']


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


## 4. Check Keys and Duplicates

Check whether the expected primary identifiers are unique.

The main entity tables should have unique IDs, while transactional tables such as order items, payments, and reviews may contain multiple rows for the same order.

In [10]:
key_checks = {
    "customers": "customer_id",
    "orders": "order_id",
    "products": "product_id",
    "sellers": "seller_id",
}

for table_name, key in key_checks.items():
    df = tables[table_name]

    print(
        f"{table_name:<15} | "
        f"rows={len(df):>8,} | "
        f"unique {key}={df[key].nunique():>8,} | "
        f"duplicates={df[key].duplicated().sum():>5,}"
    )

customers       | rows=  99,441 | unique customer_id=  99,441 | duplicates=    0
orders          | rows=  99,441 | unique order_id=  99,441 | duplicates=    0
products        | rows=  32,951 | unique product_id=  32,951 | duplicates=    0
sellers         | rows=   3,095 | unique seller_id=   3,095 | duplicates=    0


## 5. Check the Grain of Order-Level Tables

`orders` contains one row per order, but `order_items`, `order_payments`, and potentially `order_reviews` can contain multiple rows for the same order.

These tables must be checked and aggregated before joining them to avoid duplicating orders in the final ML table.

In [11]:
order_item_counts = order_items.groupby("order_id").size()
order_payment_counts = order_payments.groupby("order_id").size()
order_review_counts = order_reviews.groupby("order_id").size()

print("Order Items")
print("Maximum rows per order:", order_item_counts.max())
print("Orders with multiple rows:", (order_item_counts > 1).sum())

print("\nOrder Payments")
print("Maximum rows per order:", order_payment_counts.max())
print("Orders with multiple rows:", (order_payment_counts > 1).sum())

print("\nOrder Reviews")
print("Maximum rows per order:", order_review_counts.max())
print("Orders with multiple rows:", (order_review_counts > 1).sum())

Order Items
Maximum rows per order: 21
Orders with multiple rows: 9803

Order Payments
Maximum rows per order: 29
Orders with multiple rows: 2961

Order Reviews
Maximum rows per order: 3
Orders with multiple rows: 547


## 6. Aggregate Order Items

An order may contain multiple items. Joining the raw `order_items` table directly to `orders` would therefore create multiple rows for some orders.

Aggregate item-level information to `order_id` before performing the final join.

In [12]:
order_items_agg = (
    order_items
    .groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        unique_product_count=("product_id", "nunique"),
        unique_seller_count=("seller_id", "nunique"),
        total_item_price=("price", "sum"),
        mean_item_price=("price", "mean"),
        max_item_price=("price", "max"),
        total_freight_value=("freight_value", "sum"),
    )
    .reset_index()
)

display(order_items_agg.head())

print("Shape:", order_items_agg.shape)
print("Duplicate order IDs:", order_items_agg["order_id"].duplicated().sum())

,order_id,item_count,unique_product_count,unique_seller_count,total_item_price,mean_item_price,max_item_price,total_freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,58.90,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,239.90,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,199.00,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.99,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,199.90,199.90,18.14


Shape: (98666, 8)
Duplicate order IDs: 0


In [13]:
assert order_items_agg["order_id"].is_unique

print("Order item aggregation validated.")

Order item aggregation validated.


## 7. Aggregate Order Payments

An order can have multiple payment records. Payment information is therefore aggregated to one row per `order_id` before joining it to the order table.

In [14]:
order_payments_agg = (
    order_payments
    .groupby("order_id")
    .agg(
        payment_count=("payment_sequential", "count"),
        payment_type_count=("payment_type", "nunique"),
        max_payment_installments=("payment_installments", "max"),
        total_payment_value=("payment_value", "sum"),
    )
    .reset_index()
)

display(order_payments_agg.head())

print("Shape:", order_payments_agg.shape)
print(
    "Duplicate order IDs:",
    order_payments_agg["order_id"].duplicated().sum()
)

,order_id,payment_count,payment_type_count,max_payment_installments,total_payment_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,2,72.19
1,00018f77f2f0320c557190d7a144bdd3,1,1,3,259.83
2,000229ec398224ef6ca0657da4fc703e,1,1,5,216.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,2,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,3,218.04


Shape: (99440, 5)
Duplicate order IDs: 0


In [15]:
assert order_payments_agg["order_id"].is_unique

print("Payment aggregation validated.")

Payment aggregation validated.


## 8. Prepare Reviews



In [16]:
order_reviews_agg = (
    order_reviews
    .groupby("order_id")
    .agg(
        review_count=("review_id", "count"),
        mean_review_score=("review_score", "mean"),
        min_review_score=("review_score", "min"),
        max_review_score=("review_score", "max"),
    )
    .reset_index()
)

display(order_reviews_agg.head())

print("Shape:", order_reviews_agg.shape)
print(
    "Duplicate order IDs:",
    order_reviews_agg["order_id"].duplicated().sum()
)

,order_id,review_count,mean_review_score,min_review_score,max_review_score
0,00010242fe8c5a6d1ba2dd792cb16214,1,5.0,5,5
1,00018f77f2f0320c557190d7a144bdd3,1,4.0,4,4
2,000229ec398224ef6ca0657da4fc703e,1,5.0,5,5
3,00024acbcdf0a6daa1e931b038114c75,1,4.0,4,4
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,5.0,5,5


Shape: (98673, 5)
Duplicate order IDs: 0


In [17]:
assert order_reviews_agg["order_id"].is_unique

print("Review aggregation validated.")

Review aggregation validated.


## 9. Prepare Product and Seller Information

Products and sellers are linked through `order_items`.

The item-level table is enriched with product and seller information first, then aggregated back to one row per order.

In [18]:
print("Products columns:")
print(products.columns.tolist())

print("\nSellers columns:")
print(sellers.columns.tolist())

print("\nCategory translation columns:")
print(category_translation.columns.tolist())

Products columns:
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Sellers columns:
['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Category translation columns:
['product_category_name', 'product_category_name_english']


In [19]:
products_enriched = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

display(products_enriched.head())

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares


In [20]:
order_items_enriched = (
    order_items
    .merge(
        products_enriched[
            [
                "product_id",
                "product_category_name",
                "product_category_name_english",
                "product_weight_g",
                "product_length_cm",
                "product_height_cm",
                "product_width_cm",
            ]
        ],
        on="product_id",
        how="left"
    )
    .merge(
        sellers[
            [
                "seller_id",
                "seller_zip_code_prefix",
                "seller_city",
                "seller_state",
            ]
        ],
        on="seller_id",
        how="left"
    )
)

display(order_items_enriched.head())

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_category_name_english,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,cool_stuff,650.0,28.0,9.0,14.0,27277,volta redonda,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,pet_shop,30000.0,50.0,30.0,40.0,3471,sao paulo,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,furniture_decor,3050.0,33.0,13.0,33.0,37564,borda da mata,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,perfumery,200.0,16.0,10.0,15.0,14403,franca,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,garden_tools,3750.0,35.0,40.0,30.0,87900,loanda,PR


### Aggregate Product and Seller Information to Order Level

Because one order can contain multiple products and multiple sellers, only order-level summaries are kept here.

In [21]:
product_seller_agg = (
    order_items_enriched
    .groupby("order_id")
    .agg(
        unique_category_count=("product_category_name_english", "nunique"),
        unique_seller_state_count=("seller_state", "nunique"),
        mean_product_weight_g=("product_weight_g", "mean"),
        max_product_weight_g=("product_weight_g", "max"),
        mean_product_length_cm=("product_length_cm", "mean"),
        mean_product_height_cm=("product_height_cm", "mean"),
        mean_product_width_cm=("product_width_cm", "mean"),
    )
    .reset_index()
)

display(product_seller_agg.head())

print("Shape:", product_seller_agg.shape)
print(
    "Duplicate order IDs:",
    product_seller_agg["order_id"].duplicated().sum()
)

,order_id,unique_category_count,unique_seller_state_count,mean_product_weight_g,max_product_weight_g,mean_product_length_cm,mean_product_height_cm,mean_product_width_cm
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,650.0,650.0,28.0,9.0,14.0
1,00018f77f2f0320c557190d7a144bdd3,1,1,30000.0,30000.0,50.0,30.0,40.0
2,000229ec398224ef6ca0657da4fc703e,1,1,3050.0,3050.0,33.0,13.0,33.0
3,00024acbcdf0a6daa1e931b038114c75,1,1,200.0,200.0,16.0,10.0,15.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,3750.0,3750.0,35.0,40.0,30.0


Shape: (98666, 8)
Duplicate order IDs: 0


In [22]:
assert product_seller_agg["order_id"].is_unique

print("Product and seller aggregation validated.")

Product and seller aggregation validated.


### Check Whether Geolocation Should Be Included

Geographic information may be useful because delivery performance can depend on the location of the customer and seller.

Before including coordinates, check:

- whether ZIP-code prefixes map to multiple geolocation rows,
- how much customer and seller coverage is available,
- whether coordinates contain missing values,
- and whether geographic information can later support meaningful features such as customer–seller distance.

If geolocation is included, it must first be reduced to one row per ZIP-code prefix to avoid duplicating orders during joins.

In [23]:
geo_check = geolocation.groupby(
    "geolocation_zip_code_prefix"
).size()

print("Total rows:", len(geolocation))
print(
    "Unique ZIP prefixes:",
    geolocation["geolocation_zip_code_prefix"].nunique()
)
print("Maximum rows per ZIP prefix:", geo_check.max())
print(
    "ZIP prefixes with multiple rows:",
    (geo_check > 1).sum()
)

Total rows: 1000163
Unique ZIP prefixes: 19015
Maximum rows per ZIP prefix: 1146
ZIP prefixes with multiple rows: 17972


In [24]:
customer_geo_coverage = (
    customers["customer_zip_code_prefix"]
    .isin(geolocation["geolocation_zip_code_prefix"])
    .mean()
)

print(
    f"Customer ZIP coverage: "
    f"{customer_geo_coverage:.2%}"
)

Customer ZIP coverage: 99.72%


In [25]:
seller_geo_coverage = (
    sellers["seller_zip_code_prefix"]
    .isin(geolocation["geolocation_zip_code_prefix"])
    .mean()
)

print(
    f"Seller ZIP coverage: "
    f"{seller_geo_coverage:.2%}"
)

Seller ZIP coverage: 99.77%


In [26]:
geolocation[
    ["geolocation_lat", "geolocation_lng"]
].describe()

,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06
mean,-2.117615e+01,-4.639054e+01
std,5.715866e+00,4.269748e+00
min,-3.660537e+01,-1.014668e+02
25%,-2.360355e+01,-4.857317e+01
50%,-2.291938e+01,-4.663788e+01
75%,-1.997962e+01,-4.376771e+01
max,4.506593e+01,1.211054e+02


In [27]:
geolocation[
    ["geolocation_lat", "geolocation_lng"]
].isna().sum()

geolocation_lat    0
geolocation_lng    0
dtype: int64

### Geolocation Decision

The geolocation table contains multiple records for the same ZIP-code prefix, so a direct join would create duplicate rows.

Customer and seller ZIP codes have sufficient geolocation coverage. Geographic information is also potentially relevant to delivery performance because it can later be used to estimate the distance between customers and sellers.

Therefore, geolocation will be retained. Latitude and longitude are aggregated to one representative coordinate per ZIP-code prefix before joining.

The relationship between geographic distance and late delivery will be investigated using the training data during EDA.

##  Prepare Geolocation Information

The geolocation table contains multiple latitude/longitude records for the same ZIP-code prefix.

To preserve one row per order, geolocation is first aggregated to one row per ZIP-code prefix using the mean latitude and longitude.

In [28]:
geo_counts = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .size()
)

print("Unique ZIP prefixes:", geolocation["geolocation_zip_code_prefix"].nunique())
print("Total geolocation rows:", len(geolocation))
print("Maximum rows for one ZIP prefix:", geo_counts.max())
print("ZIP prefixes with multiple rows:", (geo_counts > 1).sum())

Unique ZIP prefixes: 19015
Total geolocation rows: 1000163
Maximum rows for one ZIP prefix: 1146
ZIP prefixes with multiple rows: 17972


In [29]:
geolocation_agg = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "mean"),
        longitude=("geolocation_lng", "mean"),
    )
    .reset_index()
)

display(geolocation_agg.head())

print("Shape:", geolocation_agg.shape)
print(
    "Duplicate ZIP prefixes:",
    geolocation_agg["geolocation_zip_code_prefix"].duplicated().sum()
)

,geolocation_zip_code_prefix,latitude,longitude
0,1001,-23.550190,-46.634024
1,1002,-23.548146,-46.634979
2,1003,-23.548994,-46.635731
3,1004,-23.549799,-46.634757
4,1005,-23.549456,-46.636733


Shape: (19015, 3)
Duplicate ZIP prefixes: 0


In [30]:
assert geolocation_agg["geolocation_zip_code_prefix"].is_unique

print("Geolocation aggregation validated.")

Geolocation aggregation validated.


In [31]:
customer_geo = geolocation_agg.rename(
    columns={
        "geolocation_zip_code_prefix": "customer_zip_code_prefix",
        "latitude": "customer_latitude",
        "longitude": "customer_longitude",
    }
)

customer_geo.head()

,customer_zip_code_prefix,customer_latitude,customer_longitude
0,1001,-23.550190,-46.634024
1,1002,-23.548146,-46.634979
2,1003,-23.548994,-46.635731
3,1004,-23.549799,-46.634757
4,1005,-23.549456,-46.636733


In [32]:
seller_geo = geolocation_agg.rename(
    columns={
        "geolocation_zip_code_prefix": "seller_zip_code_prefix",
        "latitude": "seller_latitude",
        "longitude": "seller_longitude",
    }
)

seller_geo.head()

,seller_zip_code_prefix,seller_latitude,seller_longitude
0,1001,-23.550190,-46.634024
1,1002,-23.548146,-46.634979
2,1003,-23.548994,-46.635731
3,1004,-23.549799,-46.634757
4,1005,-23.549456,-46.636733


In [33]:
customer_features = (
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state",
        ]
    ]
    .merge(
        customer_geo,
        on="customer_zip_code_prefix",
        how="left"
    )
)

display(customer_features.head())

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,customer_latitude,customer_longitude
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,-20.498489,-47.396929
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,-23.727992,-46.542848
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,-23.531642,-46.656289
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,-23.499702,-46.185233
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,-22.975100,-47.142925


In [34]:
sellers_geo_enriched = (
    sellers
    .merge(
        seller_geo,
        on="seller_zip_code_prefix",
        how="left"
    )
)

display(sellers_geo_enriched.head())

,seller_id,seller_zip_code_prefix,seller_city,seller_state,seller_latitude,seller_longitude
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,-22.893848,-47.061337
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,-22.383437,-46.947927
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,-22.909572,-43.177703
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP,-23.657242,-46.612831
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,-22.964803,-46.534419


In [35]:
order_items_enriched = (
    order_items
    .merge(
        products_enriched[
            [
                "product_id",
                "product_category_name",
                "product_category_name_english",
                "product_weight_g",
                "product_length_cm",
                "product_height_cm",
                "product_width_cm",
            ]
        ],
        on="product_id",
        how="left"
    )
    .merge(
        sellers_geo_enriched[
            [
                "seller_id",
                "seller_zip_code_prefix",
                "seller_city",
                "seller_state",
                "seller_latitude",
                "seller_longitude",
            ]
        ],
        on="seller_id",
        how="left"
    )
)

In [36]:
product_seller_agg = (
    order_items_enriched
    .groupby("order_id")
    .agg(
        unique_category_count=("product_category_name_english", "nunique"),
        unique_seller_count=("seller_id", "nunique"),
        unique_seller_state_count=("seller_state", "nunique"),

        mean_product_weight_g=("product_weight_g", "mean"),
        max_product_weight_g=("product_weight_g", "max"),
        mean_product_length_cm=("product_length_cm", "mean"),
        mean_product_height_cm=("product_height_cm", "mean"),
        mean_product_width_cm=("product_width_cm", "mean"),

        mean_seller_latitude=("seller_latitude", "mean"),
        mean_seller_longitude=("seller_longitude", "mean"),
    )
    .reset_index()
)

assert product_seller_agg["order_id"].is_unique

display(product_seller_agg.head())

,order_id,unique_category_count,unique_seller_count,unique_seller_state_count,mean_product_weight_g,max_product_weight_g,mean_product_length_cm,mean_product_height_cm,mean_product_width_cm,mean_seller_latitude,mean_seller_longitude
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,650.0,650.0,28.0,9.0,14.0,-22.496953,-44.127492
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,30000.0,30000.0,50.0,30.0,40.0,-23.565096,-46.518565
2,000229ec398224ef6ca0657da4fc703e,1,1,1,3050.0,3050.0,33.0,13.0,33.0,-22.262584,-46.171124
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,200.0,200.0,16.0,10.0,15.0,-20.553624,-47.387359
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,3750.0,3750.0,35.0,40.0,30.0,-22.929384,-53.135873


## 10. Join Everything to Orders

The final ML table is built starting from `orders`, which has one row per order.

All other tables are joined only after being reduced to one row per order where necessary.

In [37]:
customer_features = customers[
    [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
    ]
].copy()

print("Customer rows:", len(customer_features))
print(
    "Duplicate customer_id:",
    customer_features["customer_id"].duplicated().sum()
)

Customer rows: 99441
Duplicate customer_id: 0


In [38]:
ml_table = (
    orders
    .merge(
        customer_features,
        on="customer_id",
        how="left"
    )
    .merge(
        order_items_agg,
        on="order_id",
        how="left"
    )
    .merge(
        order_payments_agg,
        on="order_id",
        how="left"
    )
    .merge(
        order_reviews_agg,
        on="order_id",
        how="left"
    )
    .merge(
        product_seller_agg,
        on="order_id",
        how="left"
    )
)

print("Final shape:", ml_table.shape)
display(ml_table.head())

Final shape: (99441, 37)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,unique_category_count,unique_seller_count_y,unique_seller_state_count,mean_product_weight_g,max_product_weight_g,mean_product_length_cm,mean_product_height_cm,mean_product_width_cm,mean_seller_latitude,mean_seller_longitude
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1.0,1.0,1.0,500.0,500.0,19.0,8.0,13.0,-23.680729,-46.444238
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,1.0,1.0,400.0,400.0,19.0,13.0,19.0,-19.807681,-43.980427
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,1.0,1.0,420.0,420.0,24.0,19.0,21.0,-21.363502,-48.229601
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,1.0,1.0,1.0,450.0,450.0,30.0,10.0,20.0,-19.837682,-43.924053
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,1.0,1.0,1.0,250.0,250.0,51.0,15.0,15.0,-23.543395,-46.262086


## 11. Validate the Final ML Table


The checks verify:

- the final row count matches the original `orders` table,
- `order_id` remains unique,
- no orders were lost or unexpectedly added,
- important joined features have reasonable coverage,
- and the final table contains the columns required for the next notebook.

In [39]:
print("Original orders:")
print(f"Rows: {len(orders):,}")
print(f"Unique order IDs: {orders['order_id'].nunique():,}")

print("\nFinal ML table:")
print(f"Rows: {len(ml_table):,}")
print(f"Unique order IDs: {ml_table['order_id'].nunique():,}")
print(f"Duplicate order IDs: {ml_table['order_id'].duplicated().sum():,}")

Original orders:
Rows: 99,441
Unique order IDs: 99,441

Final ML table:
Rows: 99,441
Unique order IDs: 99,441
Duplicate order IDs: 0


In [40]:
assert ml_table["order_id"].is_unique, \
    "Validation failed: duplicate order IDs exist."

assert len(ml_table) == len(orders), \
    "Validation failed: final row count differs from orders."

assert set(ml_table["order_id"]) == set(orders["order_id"]), \
    "Validation failed: order IDs changed during joins."

print("✓ One row per order")
print("✓ Row count preserved")
print("✓ No orders lost or added")

✓ One row per order
✓ Row count preserved
✓ No orders lost or added


In [41]:
required_columns = [
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

missing_required = [
    col for col in required_columns
    if col not in ml_table.columns
]

assert not missing_required, \
    f"Missing required columns: {missing_required}"

print("✓ Required columns for Notebook 2 are available.")

✓ Required columns for Notebook 2 are available.


In [42]:
print("Final shape:", ml_table.shape)
print("\nFinal columns:")
print(ml_table.columns.tolist())

display(ml_table.head())

Final shape: (99441, 37)

Final columns:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'unique_product_count', 'unique_seller_count_x', 'total_item_price', 'mean_item_price', 'max_item_price', 'total_freight_value', 'payment_count', 'payment_type_count', 'max_payment_installments', 'total_payment_value', 'review_count', 'mean_review_score', 'min_review_score', 'max_review_score', 'unique_category_count', 'unique_seller_count_y', 'unique_seller_state_count', 'mean_product_weight_g', 'max_product_weight_g', 'mean_product_length_cm', 'mean_product_height_cm', 'mean_product_width_cm', 'mean_seller_latitude', 'mean_seller_longitude']


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,unique_category_count,unique_seller_count_y,unique_seller_state_count,mean_product_weight_g,max_product_weight_g,mean_product_length_cm,mean_product_height_cm,mean_product_width_cm,mean_seller_latitude,mean_seller_longitude
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1.0,1.0,1.0,500.0,500.0,19.0,8.0,13.0,-23.680729,-46.444238
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,1.0,1.0,400.0,400.0,19.0,13.0,19.0,-19.807681,-43.980427
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,1.0,1.0,420.0,420.0,24.0,19.0,21.0,-21.363502,-48.229601
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,1.0,1.0,1.0,450.0,450.0,30.0,10.0,20.0,-19.837682,-43.924053
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,1.0,1.0,1.0,250.0,250.0,51.0,15.0,15.0,-23.543395,-46.262086


## 12. Save the Artifact

The validated order-level dataset is saved as a Parquet artifact.

Parquet is used to preserve data types efficiently between notebooks. Notebook 2 will read this artifact directly instead of rebuilding the joins or querying the database again.

**Output:** `artifacts/01_joined/ml_orders.parquet`

In [43]:
ml_table.to_parquet(
    ARTIFACT_PATH,
    index=False
)

print("✓ Artifact saved successfully")
print("Path:", ARTIFACT_PATH)

✓ Artifact saved successfully
Path: C:\Users\CW\Documents\Projects\Dataset_Olist\artifacts\01_joined\ml_orders.parquet
